# 18 · Function Arguments and Scope

**Level 2 · Data Structures and Functions** &nbsp;|&nbsp; Previous: [17_Functions](17_Functions.ipynb) &nbsp;|&nbsp; Next: [19_Recursion_and_Function_Design](19_Recursion_and_Function_Design.ipynb)

### 🎯 Learning goals

By the end of this notebook you will be able to:

- call a function with **positional**, **keyword** and **default** arguments, and know the order rules
- write functions that accept any number of arguments with `*args` and `**kwargs`
- explain **local vs global scope** and the **LEGB** rule, and predict what a scope puzzle prints
- explain why changing a global from inside a function is usually a bad idea, and what to do instead
- predict what happens when you pass a **mutable** object to a function, and avoid the **mutable default argument** pitfall
- *(optional)* read and write simple **type hints**

In [ ]:
import sys

sys.path.insert(0, "../src")
from course_tools import check, check_output

## 1 · Why this matters

In notebook 17 you learned to package code into functions and to hand them values. So far every function took a fixed number of values in a fixed order. Real functions need more flexibility: a `greet()` that works with or without a custom greeting, a `print()` that accepts any number of things (you've been using one since notebook 01!), a function that you can call without remembering which argument comes first.

You also need to know **where names live**. A variable created inside a function is invisible outside it, and that is a feature: it lets you write a function without worrying about clashing with every other name in your program. But it also leads to some classic surprises, and a famous trap involving lists (notebook 12's aliasing lesson comes back here).

This notebook is about how information **flows into** a function (arguments), and **which names a function can see** (scope).

A quick vocabulary reminder: a **parameter** is the name in the `def` line; an **argument** is the actual value you pass in when calling.

## 2 · Positional and keyword arguments

So far you've used **positional** arguments: the first argument goes to the first parameter, the second to the second, and so on. Position decides the meaning.

In [ ]:
def divide(numerator, denominator):
    return numerator / denominator

print(divide(20, 4))
print(divide(4, 20))    # same numbers, different order, different answer

With many parameters it gets hard to remember the order. A **keyword argument** names the parameter explicitly, as `name=value`, and then the order no longer matters:

In [ ]:
print(divide(numerator=20, denominator=4))
print(divide(denominator=4, numerator=20))   # same call, different order

You can mix the two, but **positional arguments must come first**:

In [ ]:
print(divide(20, denominator=4))

You have been using keyword arguments since notebook 01: in `print("a", "b", sep="-")`, `sep` is a keyword argument. Keywords also make calls **readable**. Compare `round(3.14159, 2)` with `round(3.14159, ndigits=2)`: the second one says what the `2` means.

### Try it

Call `divide` three ways to get `5.0`: positionally, with both keywords, and with one positional plus one keyword.

In [ ]:
# your code here

## 3 · Default values

A parameter can have a **default value**, used when the caller leaves that argument out. Write it with `=` in the `def` line:

In [ ]:
def greet(name, greeting="Hello"):
    return f"{greeting}, {name}!"

print(greet("Ada"))                # uses the default
print(greet("Ada", "Good morning"))  # overrides it
print(greet("Ada", greeting="Hi"))   # overrides it, by keyword

Defaults make the common case short and the unusual case possible. Look at the built-in `print`: its `sep=" "` and `end="\n"` are just defaults.

**Rule: parameters with defaults must come after those without.** Python needs to know which arguments are required. ⚠️ **Expected to fail.**

In [ ]:
def broken(greeting="Hello", name):
    return f"{greeting}, {name}!"

Defaults are fixed **when the `def` runs**, not each time you call the function. For numbers, strings and other immutable values that makes no difference. For lists it matters a lot; we'll come back to that in section 12.

### Try it

Write `power(base, exponent=2)` that returns `base ** exponent`. Call it as `power(5)` (should give 25) and `power(2, 10)` (should give 1024).

In [ ]:
# your code here

## 4 · The order rules, all in one place

When you **define** a function, parameters go in this order: required ones, then ones with defaults. When you **call** it, positional arguments come first, then keyword ones. Each parameter may receive a value only once.

In [ ]:
def make_label(text, width=10, fill="."):
    return text + fill * (width - len(text))

print(make_label("Name"))
print(make_label("Name", 8))
print(make_label("Name", fill="*"))        # skip width, set fill
print(make_label("Name", fill="-", width=6))

Notice the third call: keywords let you **skip** a default in the middle and set only the one you care about. With positional arguments alone you'd have to repeat `10` just to reach `fill`.

## 5 · `*args`: any number of positional arguments

How does `print("a", "b", "c", "d")` accept as many arguments as you like? A parameter written with a leading `*` **collects all the extra positional arguments into a tuple**. By convention it is called `args`, but a descriptive name is better.

In [ ]:
def show(*things):
    print("I got", len(things), "arguments:", things)

show()
show("apple")
show("apple", 3, True)

Inside the function, `things` is an ordinary tuple (notebook 13), so you can loop over it, index it, or call `len()` on it. A function that adds up whatever it is given:

In [ ]:
def add_all(*numbers):
    total = 0
    for n in numbers:
        total += n
    return total

print(add_all())
print(add_all(5))
print(add_all(1, 2, 3, 4))

Regular parameters can come before `*args`. Anything **after** `*args` can only be given by keyword, which is a neat way to force clarity:

In [ ]:
def join_words(*words, separator=" "):
    return separator.join(words)

print(join_words("to", "be", "or", "not"))
print(join_words("to", "be", "or", "not", separator="-"))

### Try it

Write `largest(*numbers)` that returns the biggest of its arguments. (Hint: the built-in `max` takes a tuple... or write a loop.)

In [ ]:
# your code here

## 6 · `**kwargs`: any number of keyword arguments

The keyword twin: a parameter with a leading `**` **collects all extra keyword arguments into a dictionary** (notebook 14). The names become the keys and the values become the values. By convention it is called `kwargs`.

In [ ]:
def show_options(**options):
    print(options)
    for key, value in options.items():
        print(f"  {key} = {value}")

show_options(color="red", size=12)
show_options()

This is how a function can accept "any extra settings" without you listing them all. A profile builder:

In [ ]:
def make_profile(name, **details):
    profile = {"name": name}
    for key, value in details.items():
        profile[key] = value
    return profile

print(make_profile("Ada", born=1815, field="mathematics"))

When you use all the kinds together, the order in the `def` line is:

```
def f(required, optional=default, *args, keyword_only=default, **kwargs):
```

You will rarely need all of them at once. Reaching for `*args` and `**kwargs` is for when you genuinely want "any number of"; otherwise ordinary named parameters are clearer.

### Try it

Write `describe(**facts)` that prints one line per fact in the form `key: value`. Call it with `describe(animal="owl", legs=2)`.

In [ ]:
# your code here

## 7 · Unpacking when you call

The `*` and `**` symbols also work in the other direction, at the **call**: `*` spreads a list or tuple into positional arguments, and `**` spreads a dictionary into keyword arguments.

In [ ]:
def make_date(year, month, day):
    return f"{day}/{month}/{year}"

parts = [2026, 10, 19]
info = {"day": 5, "month": 3, "year": 2024}

print(make_date(*parts))     # same as make_date(2026, 10, 19)
print(make_date(**info))     # same as make_date(day=5, month=3, year=2024)

Think of `*` as "take the items out of the container" (just like unpacking in notebook 13). The dictionary's keys must match the parameter names exactly.

## 8 · Scope: where names live

Every name lives in a **scope**, the region of the program where it can be seen. Variables created **inside** a function are **local**: they are born when the function is called and vanish when it returns. Variables created outside any function are **global**.

In [ ]:
def make_message():
    message = "created inside"     # local
    return message

print(make_message())

⚠️ **Expected to fail.**

In [ ]:
print(message)    # the local name is gone

That error is the point: the function's internals are private. Two functions can both use a variable called `total` and never interfere. Parameters are local too.

Reading a global from inside a function works fine:

In [ ]:
TAX_RATE = 0.2

def with_tax(price):
    return price * (1 + TAX_RATE)    # reads the global

print(with_tax(100))

But **assigning** to a name inside a function creates a *new local variable*, even if a global of the same name exists. The global is untouched:

In [ ]:
x = "global"

def change():
    x = "local"        # a brand new local x
    print("inside:", x)

change()
print("outside:", x)

Picture each function call as a fresh room with its own whiteboard. Names written on the whiteboard in the room stay in the room.

### Try it

Without running it, predict what this prints. Then run it to check.

In [ ]:
n = 1

def f():
    n = 2
    return n

result = f()
print(n, result)

## 9 · The LEGB rule

When Python meets a name, it searches for it in this order and stops at the first match:

1. **L**ocal: names in the current function
2. **E**nclosing: names in any function this one is nested inside (functions can be defined inside functions; this is rare for now, and we'll see it properly in Level 6)
3. **G**lobal: names at the top level of the notebook or file
4. **B**uilt-in: Python's own names such as `print`, `len`, `max`

The initials spell **LEGB**. An example with a name defined at several levels:

In [ ]:
level = "global"

def outer():
    level = "enclosing"

    def inner():
        level = "local"
        print("inner sees:", level)

    inner()
    print("outer sees:", level)

outer()
print("top level sees:", level)

Each print finds the nearest definition. Remove the `level = "local"` line from `inner` and it would find the enclosing one; remove that as well and it finds the global.

The last stop, built-ins, explains a mistake from notebook 02:

In [ ]:
print(len("hello"))     # found in the built-in scope

len = 99               # a global named len now hides the built-in
print(len)
del len                # remove our global; the built-in is visible again
print(len("hello"))

### A function reads the global's *current* value

A function looks names up when it **runs**, not when it is defined:

In [ ]:
greeting = "Hello"

def welcome(name):
    return f"{greeting}, {name}"

print(welcome("Ada"))
greeting = "Welcome"
print(welcome("Ada"))

## 10 · Changing a global: `global`, and why to avoid it

What if you really do want a function to update a global? Assigning to it makes a local instead, and mixing the two causes a confusing error: ⚠️ **Expected to fail.**

In [ ]:
counter = 0

def increment():
    counter = counter + 1     # Python sees an assignment, so counter is local here...
    return counter

increment()

Because the function assigns to `counter`, Python treats it as local for the **whole** function, so the right-hand side reads a local that has no value yet. That's an `UnboundLocalError`.

The `global` statement tells Python "this name refers to the global one":

In [ ]:
counter = 0

def increment():
    global counter
    counter = counter + 1

increment()
increment()
print(counter)

It works, but use it sparingly. A function that secretly changes a global is hard to understand and hard to test: you can't tell what it does from its call, and it behaves differently depending on what happened before. The better habit is the one from notebook 17:

> **Pass things in as arguments, send results out with `return`.**

In [ ]:
def increment(value):
    return value + 1

counter = 0
counter = increment(counter)
counter = increment(counter)
print(counter)

Same result, but now the function is self-contained: `increment(5)` always gives `6`, whatever else is going on. Globals are fine for fixed settings that never change, such as `TAX_RATE` above (written in capitals by convention).

### Try it

The function below is meant to add `1` to a score but doesn't. Fix it using `return`, not `global`.

In [ ]:
score = 10

def add_point():
    score = score + 1

# fix the function, then call it so that score becomes 11

## 11 · Passing mutable objects

When you call a function, the parameter name is bound to **the same object** you passed: nothing is copied. You met this idea in notebook 12 as *aliasing*: two names, one list.

For an **immutable** value (number, string, tuple) this is harmless, because the object can't change. Reassigning the parameter just points the local name somewhere else:

In [ ]:
def try_to_change(number):
    number = number + 100
    return number

n = 5
try_to_change(n)
print(n)

For a **mutable** value like a list, the function can change the caller's object *in place*:

In [ ]:
def add_bonus(scores):
    scores.append(100)      # changes the caller's list

marks = [70, 80]
add_bonus(marks)
print(marks)

Nothing was returned, yet `marks` changed: `scores` and `marks` are aliases for the same list. This is occasionally what you want (that is how `list.sort()` works), but if you don't expect it, it's a nasty surprise.

Contrast **reassigning** the parameter with **mutating** it:

In [ ]:
def reassign(items):
    items = [1, 2, 3]       # the local name now points to a NEW list

def mutate(items):
    items.append(4)         # changes the object the caller also has

data = [0]
reassign(data)
print(data)
mutate(data)
print(data)

**Rule of thumb:** a function should either modify its argument in place and return `None`, or leave it alone and return a new object. Document which it is in the docstring. To leave the caller's list alone, build a new list or work on a copy (`items.copy()` or `items[:]`, from notebook 12):

In [ ]:
def with_bonus(scores):
    new_scores = scores.copy()
    new_scores.append(100)
    return new_scores

marks = [70, 80]
better = with_bonus(marks)
print(marks, better)

### Try it

Write `sorted_copy(numbers)` that returns a sorted version of the list without changing the original. (Make a copy, sort the copy.)

In [ ]:
# your code here

## 12 · The mutable default argument pitfall

Remember: a default value is created **once**, when the `def` runs, and reused for every call that doesn't supply that argument. Combine that with a mutable default and you get one of Python's most famous traps:

In [ ]:
def add_item(item, items=[]):
    items.append(item)
    return items

print(add_item("a"))
print(add_item("b"))    # we expected ['b']
print(add_item("c"))

The default list is created once and **every call shares it**. It is the aliasing lesson from notebook 12 in disguise: all calls without an `items` argument are aliases for the same list. You can even see the stored default:

In [ ]:
print(add_item.__defaults__)

**The fix:** use `None` as the default (an immutable "nothing here" marker) and create the fresh list inside the function:

In [ ]:
def add_item(item, items=None):
    if items is None:
        items = []
    items.append(item)
    return items

print(add_item("a"))
print(add_item("b"))
print(add_item("c", ["x", "y"]))

**Rule:** never use a list, dictionary or set as a default value. Use `None` and create it inside.

### Try it

Write `tag_all(word, tags=None)` that appends `word` to `tags` (a fresh list if none was given) and returns it. Call it twice without `tags` and confirm the results are separate lists.

In [ ]:
# your code here

## 13 · Type hints *(optional)*

*This section is optional: type hints never change how your code runs, so you can skip it and come back later.*

Python doesn't ask you to declare what type a parameter should be. But you can **annotate** the parameters and the return value as documentation for the reader (and for editors, which can then warn you about mistakes):

In [ ]:
def area(width: float, height: float) -> float:
    return width * height

def shout(text: str, times: int = 1) -> str:
    return (text.upper() + "! ") * times

print(area(3, 4.5))
print(shout("hey", 2))

The pattern is `parameter: type`, with `= default` after the hint, and `-> type` before the colon for the return value. Common types: `int`, `float`, `str`, `bool`, `list`, `dict`, `tuple`, `set`, and `None` for "returns nothing".

Python does **not** enforce them. This runs without complaint:

In [ ]:
print(area("ab", 3))      # a str is not a float, but Python doesn't check

The hints are stored on the function and can be looked at:

In [ ]:
print(area.__annotations__)

Tools like `mypy` or your editor can read the hints and catch mistakes before you run the code. (Later levels cover more advanced hints, such as lists of ints.) For now: a hint is a small, checked-by-humans comment about what goes in and what comes out.

## 14 · Common mistakes

### 1. Positional argument after a keyword argument

⚠️ **Expected to fail.**

In [ ]:
print(divide(numerator=20, 4))

Once you start using keywords, the rest must be keywords too.

### 2. Missing arguments

⚠️ **Expected to fail.**

In [ ]:
greet()

`TypeError: greet() missing 1 required positional argument: 'name'`. Read it: it names the function and the parameter.

### 3. Too many arguments

⚠️ **Expected to fail.**

In [ ]:
greet("Ada", "Hi", "extra")

### 4. The same parameter given twice

⚠️ **Expected to fail.**

In [ ]:
greet("Ada", name="Grace")

The first positional argument already went to `name`.

### 5. Misspelling a keyword

⚠️ **Expected to fail.**

In [ ]:
greet("Ada", greting="Hi")

### 6. Expecting a local variable to exist outside

You saw this in section 8: `NameError`. If you need a value from a function, `return` it.

### 7. Assigning to a global without declaring it

You saw this in section 10: `UnboundLocalError`. Prefer arguments and `return` over `global`.

### 8. Mutable defaults

`def f(x, items=[])` shares one list between all calls (section 12). Use `None`.

### 9. Calling a function by name without parentheses

`greet` is the function itself; `greet("Ada")` runs it. Printing `greet` just shows something like `<function greet at 0x...>`.

### 10. Naming a parameter after a built-in

A parameter called `list`, `sum` or `max` hides the built-in inside the function (LEGB again!). Use `numbers`, `total`, `largest`.

In [ ]:
def average(list):            # works, but...
    return sum(list) / len(list)

print(average([2, 4, 6]))

def bad(list):
    return list([1, 2])        # the built-in list is hidden here

⚠️ **Expected to fail.**

In [ ]:
bad("anything")

## 15 · Exercises

### Exercise 18.1 · Warm-up: a greeting with a default

Complete `greet(name, greeting)` so that it returns `"<greeting>, <name>!"`, and give `greeting` a default value of `"Hello"`.

In [ ]:
def greet(name, greeting=None):    # replace None with a sensible default
    # your code here
    pass

In [ ]:
check(greet("Ada"), "Hello, Ada!")
check(greet("Ben", "Hi"), "Hi, Ben!")
check(greet("Cy", greeting="Welcome"), "Welcome, Cy!")

### Exercise 18.2 · Warm-up: keywords in any order

The function `divide` is defined for you. Using **keyword arguments** for both parameters, store `20 / 4` in `r1` by writing the denominator first. Then, using one positional and one keyword argument, store `9 / 3` in `r2`.

In [ ]:
def divide(numerator, denominator):
    return numerator / denominator

r1 = None
r2 = None

In [ ]:
check(r1, 5.0)
check(r2, 3.0)

### Exercise 18.3 · Warm-up: any number of arguments

Write `total(*numbers)` that returns the sum of all its arguments (`0` if there are none). Do not use the built-in `sum`; use a loop.

In [ ]:
def total(*numbers):
    # your code here
    pass

In [ ]:
check(total(), 0)
check(total(5), 5)
check(total(1, 2, 3, 4), 10)

### Exercise 18.4 · Practice: predict the scope

Read each puzzle and decide what it prints **without running it**. Store each printed line as a string in the variables below (the `print` shows values separated by a single space). Then run the code in a scratch cell to check yourself.

**Puzzle A**

```python
x = "global"

def f():
    x = "local"
    return x

y = f()
print(x, y)
```

**Puzzle B**

```python
count = 0

def bump(count):
    count = count + 1
    return count

bump(count)
bump(count)
print(count)
```

**Puzzle C**

```python
total = 10

def add(n):
    return total + n

total = 100
print(add(5))
```

In [ ]:
puzzle_a = None    # a string such as "word word"
puzzle_b = None    # a number
puzzle_c = None    # a number

In [ ]:
check(puzzle_a, "global local")
check(puzzle_b, 0)
check(puzzle_c, 105)

### Exercise 18.5 · Practice: fix the score

This is the function from section 10. It tries to change a global without declaring it. Rewrite it as `add_point(score)`, which **returns** the new score (do not use `global`). Then update the variable `score` by calling it twice, so that `score` ends up as `12`.

In [ ]:
def add_point(score):
    # your code here
    pass

score = 10
# call add_point twice, keeping the result each time

In [ ]:
check(score, 12)
check(add_point(0), 1)

### Exercise 18.6 · Practice: counting calls with `global`

Sometimes a global really is the right tool. Write `record_call()` that adds `1` to the global `calls` each time it is called (you will need `global`). Then call it three times.

In [ ]:
calls = 0

def record_call():
    # your code here
    pass

# call record_call three times

In [ ]:
check(calls, 3)

### Exercise 18.7 · Practice: fix the mutable default

This function has the bug from section 12: every call shares one list. Fix it so each call without `items` starts with a fresh empty list, while a list passed in is still used.

In [ ]:
def add_item(item, items=[]):
    items.append(item)
    return items

In [ ]:
first = add_item("a")
second = add_item("b")
mine = add_item("z", ["y"])
check([first, second, mine], [["a"], ["b"], ["y", "z"]])

### Exercise 18.8 · Practice: don't touch my list

Write `doubled(numbers)` that returns a **new** list with every number doubled, leaving the original list unchanged. Build the new list with a loop and `append`.

In [ ]:
def doubled(numbers):
    # your code here
    pass

nums = [1, 2, 3]
result = doubled(nums)

In [ ]:
check([result, nums], [[2, 4, 6], [1, 2, 3]], hint="The original list must still be [1, 2, 3].")

### Exercise 18.9 · Practice: a function with sensible defaults

A courier charges `rate` per kilogram, but never less than a `minimum` fee. Write `shipping_cost(weight, rate=2.5, minimum=5.0)` that returns the larger of `weight * rate` and `minimum`. Choose the defaults shown so that most callers only need to pass the weight.

In [ ]:
def shipping_cost(weight, rate=None, minimum=None):    # replace the Nones with the defaults
    # your code here
    pass

In [ ]:
check(shipping_cost(1), 5.0)
check(shipping_cost(4), 10.0)
check(shipping_cost(4, rate=3.0), 12.0)
check(shipping_cost(1, minimum=0.0), 2.5)
check(shipping_cost(10, 2.0, 25.0), 25.0)

### Exercise 18.10 · Practice: unpack at the call

`make_date` is defined for you. Using `*` and `**` (and the data below), store the results of the two calls in `d1` and `d2`.

In [ ]:
def make_date(year, month, day):
    return f"{day}/{month}/{year}"

parts = [2026, 10, 19]
info = {"day": 5, "month": 3, "year": 2024}

d1 = None    # use parts
d2 = None    # use info

In [ ]:
check(d1, "19/10/2026")
check(d2, "5/3/2024")

### Exercise 18.11 · Stretch: collect keyword arguments

Write `make_profile(name, **details)` that returns a dictionary with the key `"name"` plus one entry per extra keyword argument.

In [ ]:
def make_profile(name, **details):
    # your code here
    pass

In [ ]:
check(make_profile("Ada", age=36, city="London"), {"name": "Ada", "age": 36, "city": "London"})
check(make_profile("Bo"), {"name": "Bo"})

### Exercise 18.12 · Stretch: flexible `greet_all`

Write `greet_all(*names, greeting="Hello")` that returns a **list** of greetings, one per name, each in the form `"<greeting>, <name>!"`. `greeting` can only be given by keyword. With no names it returns an empty list.

In [ ]:
def greet_all(*names, greeting="Hello"):
    # your code here
    pass

In [ ]:
check(greet_all("Ann", "Bo"), ["Hello, Ann!", "Hello, Bo!"])
check(greet_all("Cy", greeting="Hi"), ["Hi, Cy!"])
check(greet_all(), [])

### Exercise 18.13 · Stretch *(optional)*: add type hints

Add type hints to `bmi`: both parameters are `float` and it returns a `float`. (This exercise belongs to the optional section 13.)

In [ ]:
def bmi(weight, height):
    return weight / height ** 2

In [ ]:
check(bmi.__annotations__, {"weight": float, "height": float, "return": float})

## 16 · Recap and what's next

- **Positional** arguments are matched by position; **keyword** arguments by name (`name=value`) and may come in any order, but only **after** the positional ones.
- **Default values** (`def f(x, y=10)`) make arguments optional; parameters with defaults come last.
- **`*args`** collects extra positional arguments into a **tuple**; **`**kwargs`** collects extra keyword arguments into a **dict**. At a call, `*` and `**` unpack a sequence or a dictionary.
- **Scope:** names assigned inside a function are **local**. Python looks names up in the order **L**ocal, **E**nclosing, **G**lobal, **B**uilt-in (**LEGB**).
- Prefer **arguments and `return`** over `global`.
- Arguments are not copied: a function that mutates a **mutable** argument changes the caller's object. Return a new object, or copy first, if you don't want that.
- **Never use a mutable default** (`items=[]`); use `None` and create the object inside.
- *(Optional)* **Type hints** (`def f(x: int) -> str`) document your intent but are not enforced.

### What's next

Notebook 19, **Recursion and Function Design**, closes Level 2: functions that call themselves *(optional)*, and how to break a larger problem into small, single-purpose functions.